# Land Use Regression

Land use regression (LUR) experiment: varying LCD fraction with OLS, GAM, and RF models.

**Feature selection** uses Spearman-ordered sequential forward selection with a condition number stopping criterion: features are ranked by |Spearman correlation with $y$| and added greedily; a candidate is skipped if it would push the condition number of the feature matrix above `condition_threshold`.


In [ ]:
import json
from pathlib import Path

import dask
import numpy as np
import pandas as pd
import shap
from dask.diagnostics import ProgressBar
from flaml.default import RandomForestRegressor
from focalpy.diagnostics import moran_residuals
from focalpy.model_selection import _as_coordinate_array
from interpret.glassbox import ExplainableBoostingRegressor
from libpysal.weights import KNN
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skops import io as sio

from zurich_uhi_lur import regr_utils
from zurich_uhi_lur.regr_utils import SpearmanSFSSelector

In [ ]:
# input data paths
stations_gdf_filepath = "../data/processed/stations.gpkg"
station_features_filepath = "../data/processed/stations-features.csv"
ts_df_filepath = "../data/processed/ts-df.csv"

# output
models_dir = "../models"
dst_results_filepath = "../data/processed/lur-results.csv"
dst_importance_filepath = "../data/processed/lur-importance.csv"
dst_obs_pred_filepath = "../data/processed/lur-obs-pred.csv"
dst_baseline_moran_filepath = "../data/processed/baseline-moran.json"

# experiment settings
target_hour = 21
n_reps = 5
seed = 0
frac_step = 0.33
fracs = [0, 0.33, 0.66, 1]
n_splits = 3
condition_threshold = 10.0
rsa_permutations = 999

# dask parallelism
n_dask_workers = 4

## Data loading


In [ ]:
station_features_gdf = regr_utils.get_site_features_gdf(
    station_features_filepath, stations_gdf_filepath
)
# Station 534 is co-located with Zürich-Kaserne (identical features, distorts KNN
# spatial weights for Moran I), so it is excluded from training and all diagnostics.
station_features_gdf = station_features_gdf[station_features_gdf.index != "534"]
aws_gdf = station_features_gdf[station_features_gdf["source"] == "AWS"]
lcd_gdf = station_features_gdf[station_features_gdf["source"] == "LCD"]
station_features_gdf.head()

,source,geometry,building_volume_sum_100,building_volume_sum_250,building_volume_sum_500,tree_sum_100,tree_sum_250,tree_sum_500,elevation_0,lake_distance_0,aspect_mean_250,aspect_mean_500,slope_mean_250,slope_mean_500,topographic_position_index_mean_250,topographic_position_index_mean_500
station_id,,,,,,,,,,,,,,,,
KLO,AWS,POINT (2682711 1259339),0.047963,4.796252e-02,4.239974e+03,0.0,0.0,2091.0,426.10724,12548.878257,154.824131,145.856184,1.006813,1.756003,0.000013,0.000094
REH,AWS,POINT (2681433 1253548),53579.327844,1.289910e+05,3.275898e+05,690.0,5862.0,29177.0,443.39940,7037.797441,177.117733,174.696470,2.149442,3.195710,0.000452,-0.000174
SMA,AWS,POINT (2685118 1248066),62952.852086,3.301923e+05,1.019969e+06,4994.0,31862.0,125308.0,555.86786,2046.655244,211.169714,210.835274,10.137564,10.919506,0.000075,-0.000143
Zürich-Kaserne,AWS,POINT (2682450 1247990),263396.449315,2.060322e+06,5.209396e+06,1295.0,5673.0,26116.0,406.53400,1495.165182,171.997938,170.120588,2.356884,3.759559,0.000025,0.000003
Dübendorf-Empa,AWS,POINT (2688300 1251105),113076.299845,6.894265e+05,1.611062e+06,1483.0,9898.0,57244.0,430.77298,6439.998871,193.932043,192.267062,5.066118,5.500230,-0.000452,-0.000036


In [ ]:
ts_df = (
    pd.read_csv(ts_df_filepath, index_col=["heatwave", "time"], parse_dates=["time"])
    .rename_axis(columns=station_features_gdf.index.name)
    .droplevel("heatwave")
)
ts_df = ts_df.loc[ts_df.index.get_level_values("time").hour == target_hour]
ts_df.head()

station_id,KLO,REH,SMA,Zürich-Kaserne,Dübendorf-Empa,A10,A7,B12,B3,B6,...,2680,2681,2682,2683,2688,2689,2695,2696,2697,2810
time,,,,,,,,,,,,,,,,,,,,,
2022-06-16 21:00:00,19.100000,19.566667,21.716667,27.5,24.9,NaN,NaN,NaN,22.225,21.825,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-06-17 21:00:00,18.750000,18.566667,21.016667,26.9,23.7,NaN,NaN,NaN,22.125,21.275,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-06-18 21:00:00,20.583333,20.500000,23.400000,29.7,26.0,NaN,NaN,NaN,24.250,24.075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-06-19 21:00:00,25.283333,26.533333,26.416667,32.3,30.4,NaN,NaN,NaN,27.200,27.525,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-06-20 21:00:00,22.466667,22.583333,22.100000,26.3,25.0,NaN,NaN,NaN,23.175,24.100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Baseline spatial autocorrelation of the target


In [ ]:
y_station = (
    ts_df[station_features_gdf.index]
    .apply(lambda row: row - row.mean(), axis="columns")
    .stack()
    .droplevel("time")
)
station_gser = station_features_gdf["geometry"]
groups = y_station.index

coords = _as_coordinate_array(station_gser.loc[np.unique(np.asarray(groups))])
w = KNN(coords, k=4)
w.transform = "r"

baseline_moran = moran_residuals(
    y_station.values,
    w,
    groups=groups,
    permutations=rsa_permutations,
)
print(
    f"Baseline Moran's I of y: {baseline_moran['I']:.3f} "
    f"(p = {baseline_moran['p_sim']:.3f})"
)

Baseline Moran's I of y: 0.584 (p = 0.001)


## Model pipelines

Pipeline: `StandardScaler → SpearmanSFSSelector → Model`

Features are scaled first so that Spearman correlation and condition number
are computed on a comparable scale across features.


In [ ]:
def make_pipeline(model):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("fsel", SpearmanSFSSelector(condition_threshold=condition_threshold)),
            ("model", model),
        ]
    )


def _ols_importances(pipe, X, selected):
    return dict(zip(selected, pipe.named_steps["model"].coef_))


def _signed_shap_importances(pipe, X, selected):
    """sign(Spearman(x_j, SHAP_j)) × mean(|SHAP_j|).

    Mean SHAP is identically zero (it equals mean(pred) - E[pred] = 0), so
    directional importance is recovered by taking the sign from the
    Spearman rank-correlation between each feature and its SHAP values, then
    scaling by the mean absolute SHAP.

    For EBM (additive model, interactions=0): eval_terms() returns exact
    per-sample term contributions, which are the SHAP values.
    For RF: shap.TreeExplainer is used.
    """
    X_transformed = pipe[:-1].transform(X)
    model = pipe.named_steps["model"]

    if isinstance(model, ExplainableBoostingRegressor):
        # eval_terms returns (n_samples, n_terms); with interactions=0,
        # n_terms == n_features
        shap_vals = np.asarray(model.eval_terms(X_transformed))
    else:
        shap_vals = shap.TreeExplainer(model)(
            X_transformed
        ).values  # (n_samples, n_features)

    return {
        feat: float(
            np.sign(spearmanr(X_transformed[:, j], shap_vals[:, j]).statistic)
            * np.abs(shap_vals[:, j]).mean()
        )
        for j, feat in enumerate(selected)
    }


# (pipeline factory, importance extractor) per model
MODELS = {
    "OLS": (lambda: make_pipeline(LinearRegression()), _ols_importances),
    "GAM": (
        lambda: make_pipeline(
            ExplainableBoostingRegressor(interactions=0, random_state=seed)
        ),
        _signed_shap_importances,
    ),
    "RF": (lambda: make_pipeline(RandomForestRegressor()), _signed_shap_importances),
}

## Experiment: varying LCD fraction


In [ ]:
rng = np.random.default_rng(seed)
kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

station_samples = []
for frac in fracs:
    if np.isclose(frac, 1):
        gdf = pd.concat([aws_gdf, lcd_gdf]).drop(columns="source")
        station_samples.append((frac, 0, gdf))
    else:
        for rep in range(n_reps):
            gdf = pd.concat(
                [aws_gdf, lcd_gdf.sample(frac=frac, random_state=rng)]
            ).drop(columns="source")
            station_samples.append((frac, rep, gdf))


def _run_trial(frac, rep, station_gdf, model_name, get_pipeline, importance_fn):
    """Single (frac, rep, model) trial."""
    station_gser = station_gdf["geometry"]
    y_with_time = (
        ts_df[station_gdf.index]
        .apply(lambda row: row - row.mean(), axis="columns")
        .stack()
    )
    y_ser = y_with_time.droplevel("time")
    X_df = station_gdf.loc[y_ser.index].drop(columns="geometry")
    groups = y_ser.index

    scored = regr_utils.score_pipeline(
        get_pipeline(),
        X_df,
        y_ser,
        cv=kf,
        station_gser=station_gser,
        groups=groups,
        rsa_permutations=rsa_permutations,
        importance_fn=importance_fn,
    )
    return {
        "frac": frac,
        "rep": rep,
        "model": model_name,
        **scored,
        "y_obs": y_ser.values,
        "station_ids": np.asarray(groups),
    }


tasks = [
    dask.delayed(_run_trial)(frac, rep, gdf, name, factory, imp_fn)
    for frac, rep, gdf in station_samples
    for name, (factory, imp_fn) in MODELS.items()
]

with ProgressBar():
    results = list(dask.compute(*tasks, num_workers=n_dask_workers))

# Expand frac=1 results to n_reps
results = [
    {**r, "rep": rep}
    for r in results
    for rep in (range(n_reps) if np.isclose(r["frac"], 1) else [r["rep"]])
]

_SCALAR = ["frac", "rep", "model", "r2", "r2_cv", "moran_I", "moran_p_sim"]
results_df = pd.DataFrame([{k: r.get(k) for k in _SCALAR} for r in results])

importance_df = pd.DataFrame(
    [
        {
            "frac": r["frac"],
            "rep": r["rep"],
            "model": r["model"],
            "feature": f,
            "value": v,
        }
        for r in results
        for f, v in r.get("importances", {}).items()
    ]
)

# Build obs-pred DataFrame — one row per (station_id × timestep) per trial
source_map = pd.concat(
    [
        aws_gdf[[]].assign(source="AWS"),
        lcd_gdf[[]].assign(source="LCD"),
    ]
)["source"]

obs_pred_df = pd.DataFrame(
    [
        {
            "frac": r["frac"],
            "rep": r["rep"],
            "model": r["model"],
            "station_id": sid,
            "obs": float(obs_val),
            "pred": float(pred_val),
            "source": source_map.get(sid, "LCD"),
        }
        for r in results
        for sid, obs_val, pred_val in zip(r["station_ids"], r["y_obs"], r["y_pred_cv"])
    ]
)

frac_pipelines = {
    frac: {
        r["model"]: r["pipe"]
        for r in results
        if np.isclose(r["frac"], frac) and r["rep"] == 0
    }
    for frac in fracs
}
final_pipelines = frac_pipelines[fracs[-1]]
final_features = {
    r["model"]: r["selected_features"]
    for r in results
    if np.isclose(r["frac"], fracs[-1]) and r["rep"] == 0
}

[########################################] | 100% Completed | 34m 31s


## Save results


In [ ]:
models_path = Path(models_dir)

for model_name, pipe in final_pipelines.items():
    sio.dump(pipe, models_path / f"pipeline-{model_name}.skops")

results_df.to_csv(dst_results_filepath, index=False)
importance_df.to_csv(dst_importance_filepath, index=False)
obs_pred_df.to_csv(dst_obs_pred_filepath, index=False)

# Save all pipelines
for frac in frac_pipelines:
    for model_name, pipe in frac_pipelines[frac].items():
        sio.dump(pipe, models_path / f"pipeline-frac{frac}-{model_name}.skops")

# Save baseline Moran's I
with open(dst_baseline_moran_filepath, "w") as f:
    json.dump(
        {"I": float(baseline_moran["I"]), "p_sim": float(baseline_moran["p_sim"])}, f
    )